In [56]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

In [57]:
sales      = pd.read_csv(f"../data/processed/sales.csv", parse_dates=["Date"])
promotions = pd.read_csv(f"../data/processed/promotions.csv",
                             parse_dates=["start_date", "end_date"])
web        = pd.read_csv(f"../data/processed/web_traffic.csv", parse_dates=["date"])
sample_sub = pd.read_csv(f"../data/processed/sample_submission.csv", parse_dates=["Date"])

print(f"Train rows: {len(sales):,} ({sales['Date'].min().date()} => {sales['Date'].max().date()})")
print(f"Test rows: {len(sample_sub):,} ({sample_sub['Date'].min().date()} => {sample_sub['Date'].max().date()})")
print(f"Promotions: {len(promotions):,}")
print(f"Web traffic: {len(web):,}")
sales.head()

Train rows: 3,833 (2012-07-04 => 2022-12-31)
Test rows: 548 (2023-01-01 => 2024-07-01)
Promotions: 50
Web traffic: 3,652


,Date,Revenue,COGS,is_holiday,log_Revenue,log_COGS
0,2012-07-04,5123547.94,3982991.19,0,15.449358,15.197544
1,2012-07-05,2751773.45,2150580.23,0,14.827757,14.581249
2,2012-07-06,3054029.42,2517632.84,0,14.931973,14.738830
3,2012-07-07,2667930.94,2108246.62,0,14.796814,14.561368
4,2012-07-08,2360851.90,1808622.79,0,14.674534,14.408077


## Feature Engineering

In [58]:
def build_features(df: pd.DataFrame,
                   promotions: pd.DataFrame,
                   web: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").reset_index(drop=True).copy()

    # Calendar
    df["dow"]       = df["Date"].dt.dayofweek
    df["month"]     = df["Date"].dt.month
    df["dom"]       = df["Date"].dt.day
    df["woy"]       = df["Date"].dt.isocalendar().week.astype(int)
    df["quarter"]   = df["Date"].dt.quarter
    df["is_weekend"]= df["dow"].isin([5, 6]).astype(int)
    df["year"]      = df["Date"].dt.year
    df["year_norm"] = (df["year"] - 2012) / 10.0 # xu huong tang truong

    # Tet
    tet_dates = pd.to_datetime([
        "2012-01-23","2013-02-10","2014-01-31","2015-02-19","2016-02-08",
        "2017-01-28","2018-02-16","2019-02-05","2020-01-25","2021-02-12",
        "2022-02-01","2023-01-22","2024-02-10","2025-01-29"
    ])
    df["days_to_tet"] = df["Date"].apply(
        lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=60)
    df["pre_tet_7d"]  = df["Date"].apply(
        lambda x: int(any(0  < (t - x).days <= 7  for t in tet_dates)))
    df["pre_tet_14d"] = df["Date"].apply(
        lambda x: int(any(7  < (t - x).days <= 14 for t in tet_dates)))
    df["pre_tet_30d"] = df["Date"].apply(
        lambda x: int(any(14 < (t - x).days <= 30 for t in tet_dates)))
    df["on_tet"]      = df["Date"].apply(
        lambda x: int(any(abs((x - t).days) <= 2  for t in tet_dates)))
    df["post_tet"]    = df["Date"].apply(
        lambda x: int(any(0  < (x - t).days <= 7  for t in tet_dates)))

    # Holidays
    fixed_holidays = []
    for y in range(2012, 2026):
        fixed_holidays += [f"{y}-01-01", f"{y}-04-30", f"{y}-05-01", f"{y}-09-02"]
    gioto = ["2012-03-31","2013-04-19","2014-04-09","2015-04-28","2016-04-16",
             "2017-04-06","2018-04-25","2019-04-14","2020-04-02","2021-04-21",
             "2022-04-10","2023-04-29","2024-04-18","2025-04-07"]
    all_holidays = pd.to_datetime(fixed_holidays + gioto)
    df["is_holiday"] = df["Date"].apply(
        lambda x: int(min(abs((x - h).days) for h in all_holidays) <= 3))

    # Promotion
    df["promo_count"] = 0
    for _, row in promotions.iterrows():
        mask = (df["Date"] >= row["start_date"]) & (df["Date"] <= row["end_date"])
        df.loc[mask, "promo_count"] += 1
    df["promo_active"]    = (df["promo_count"] > 0).astype(int)
    df["promo_intensity"] = df["promo_count"].clip(upper=5)
    df["post_promo"]      = ((df["promo_active"].shift(1).fillna(0) == 1) &
                              (df["promo_active"] == 0)).astype(int)

    # Web traffic
    daily_web = web.groupby("date")["sessions"].sum().reset_index()
    df = df.merge(daily_web, left_on="Date", right_on="date", how="left").drop(columns=["date"])
    df["sessions"] = df["sessions"].fillna(df["sessions"].median())

    roll30_sess = df["sessions"].shift(1).rolling(30).mean()
    df["sessions_lag7"]       = df["sessions"].shift(7)
    df["sessions_lag14"]      = df["sessions"].shift(14)
    df["sessions_roll7_mean"] = df["sessions"].shift(1).rolling(7).mean()
    df["sessions_vs_avg"]     = (df["sessions"] / roll30_sess).fillna(1.0)
    df["sessions_spike"]      = (df["sessions_vs_avg"] > 1.5).astype(int)

    # Revenue lags & rolling
    rev = df["Revenue"]
    for lag in [1, 2, 3, 7, 14, 21, 28, 30, 60, 90, 365]:
        df[f"rev_lag_{lag}"] = rev.shift(lag)

    # EWM (Exponentially Weighted Mean) - avg uu tien ngay gan hon
    for span in [7, 14, 30]:
        df[f"rev_ewm{span}"] = rev.shift(1).ewm(span=span, adjust=False).mean()

    for w in [7, 14, 30, 90]:
        df[f"rev_roll{w}_mean"] = rev.shift(1).rolling(w).mean()
        df[f"rev_roll{w}_std"]  = rev.shift(1).rolling(w).std()
        df[f"rev_roll{w}_max"]  = rev.shift(1).rolling(w).max()
        df[f"rev_roll{w}_min"]  = rev.shift(1).rolling(w).min()

    # Year-over-Year - So sanh cung ki nam truoc
    df["rev_yoy_lag365"] = rev.shift(365)
    df["rev_yoy_ratio"]  = (rev.shift(1) / rev.shift(366).replace(0, np.nan)
                            ).fillna(1.0).clip(0.1, 10)

    # COGS lags 
    cogs = df["COGS"]
    for lag in [1, 7, 30, 365]:
        df[f"cogs_lag_{lag}"] = cogs.shift(lag)
    df["cogs_roll7_mean"]  = cogs.shift(1).rolling(7).mean()
    df["cogs_roll30_mean"] = cogs.shift(1).rolling(30).mean()

    # Revenue/COGS ratio (gross margin proxy) - Nhap: 100, Ban 140 -> Tile 1.4
    # The hien hieu qua kinh doanh
    df["rev_cogs_ratio_lag1"] = (
        rev.shift(1) / cogs.shift(1).replace(0, np.nan)
    ).fillna(1.4).clip(0.5, 5)

    return df

print("build_features")

build_features


### Feature engineering - train data

In [59]:
df_feat = build_features(sales.copy(), promotions, web)

EXCLUDE = {"Date", "Revenue", "COGS", "log_Revenue", "log_COGS",
           "sessions", "promo_count", "promo_active"}

feature_cols = [c for c in df_feat.columns if c not in EXCLUDE]

print(f"Shape:{df_feat.shape}")
print(f"Total features: {len(feature_cols)}")
print(f"\nFeatures list:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")

Shape:(3833, 69)
Total features: 61

Features list:
   1. is_holiday
   2. dow
   3. month
   4. dom
   5. woy
   6. quarter
   7. is_weekend
   8. year
   9. year_norm
  10. days_to_tet
  11. pre_tet_7d
  12. pre_tet_14d
  13. pre_tet_30d
  14. on_tet
  15. post_tet
  16. promo_intensity
  17. post_promo
  18. sessions_lag7
  19. sessions_lag14
  20. sessions_roll7_mean
  21. sessions_vs_avg
  22. sessions_spike
  23. rev_lag_1
  24. rev_lag_2
  25. rev_lag_3
  26. rev_lag_7
  27. rev_lag_14
  28. rev_lag_21
  29. rev_lag_28
  30. rev_lag_30
  31. rev_lag_60
  32. rev_lag_90
  33. rev_lag_365
  34. rev_ewm7
  35. rev_ewm14
  36. rev_ewm30
  37. rev_roll7_mean
  38. rev_roll7_std
  39. rev_roll7_max
  40. rev_roll7_min
  41. rev_roll14_mean
  42. rev_roll14_std
  43. rev_roll14_max
  44. rev_roll14_min
  45. rev_roll30_mean
  46. rev_roll30_std
  47. rev_roll30_max
  48. rev_roll30_min
  49. rev_roll90_mean
  50. rev_roll90_std
  51. rev_roll90_max
  52. rev_roll90_min
  53. rev_yoy_la

## Walk-Forward Cross Validation

In [60]:
def get_lgb_params(target="Revenue"):
    return dict(
        n_estimators      = 3000,
        learning_rate     = 0.02,
        num_leaves        = 63,
        max_depth         = -1,
        min_child_samples = 20,
        feature_fraction  = 0.8,
        bagging_fraction  = 0.8,
        bagging_freq      = 5,
        reg_alpha         = 0.1,
        reg_lambda        = 0.2,
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )

def walk_forward_cv(df_train: pd.DataFrame, feature_cols: list,
                    target: str = "Revenue",
                    n_splits: int = 5,
                    min_train_days: int = 365) -> tuple:
    df_train = df_train.dropna(subset=feature_cols + [target]).reset_index(drop=True)
    n = len(df_train)
    fold_size = (n - min_train_days) // n_splits

    oof_preds = np.zeros(n)
    oof_mask  = np.zeros(n, dtype=bool)

    for i in range(n_splits):
        cut     = min_train_days + i * fold_size
        val_end = cut + fold_size if i < n_splits - 1 else n

        X_tr  = df_train.iloc[:cut][feature_cols]
        y_tr  = df_train.iloc[:cut][target]
        X_val = df_train.iloc[cut:val_end][feature_cols]
        y_val = df_train.iloc[cut:val_end][target]

        model = lgb.LGBMRegressor(**get_lgb_params(target))
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(period=-1)])

        preds = model.predict(X_val)
        mae_  = mean_absolute_error(y_val, preds)
        rmse_ = np.sqrt(mean_squared_error(y_val, preds))
        r2_   = r2_score(y_val, preds)
        print(f"  Fold {i+1} | train={cut:,} | val={val_end-cut:,} "
              f"| MAE={mae_:,.0f} | RMSE={rmse_:,.0f} | R²={r2_:.4f}")

        oof_preds[cut:val_end] = preds
        oof_mask[cut:val_end]  = True

    # OOF aggregate
    y_oof = df_train[target].values[oof_mask]
    p_oof = oof_preds[oof_mask]
    oof_mae  = mean_absolute_error(y_oof, p_oof)
    oof_rmse = np.sqrt(mean_squared_error(y_oof, p_oof))
    oof_r2   = r2_score(y_oof, p_oof)
    print(f"\nOOF: MAE={oof_mae:,.0f} | RMSE={oof_rmse:,.0f} | R²={oof_r2:.4f}\n")
    return oof_preds, oof_mask

print("walk_forward_cv")

walk_forward_cv


In [61]:
print("Walk-forward CV — REVENUE")
oof_rev_preds, oof_rev_mask = walk_forward_cv(
    df_feat, feature_cols, target="Revenue", n_splits=5
)

Walk-forward CV — REVENUE
  Fold 1 | train=365 | val=620 | MAE=837,866 | RMSE=1,182,591 | R²=0.7638
  Fold 2 | train=985 | val=620 | MAE=987,537 | RMSE=1,454,747 | R²=0.7920
  Fold 3 | train=1,605 | val=620 | MAE=845,923 | RMSE=1,128,137 | R²=0.8320
  Fold 4 | train=2,225 | val=620 | MAE=518,272 | RMSE=724,617 | R²=0.7851
  Fold 5 | train=2,845 | val=623 | MAE=546,263 | RMSE=758,227 | R²=0.7854

OOF: MAE=746,978 | RMSE=1,084,856 | R²=0.8333



In [62]:
print("Walk-forward CV — COGS")
oof_cogs_preds, oof_cogs_mask = walk_forward_cv(
    df_feat, feature_cols, target="COGS", n_splits=5
)

Walk-forward CV — COGS
  Fold 1 | train=365 | val=620 | MAE=715,134 | RMSE=1,011,001 | R²=0.7607
  Fold 2 | train=985 | val=620 | MAE=855,597 | RMSE=1,236,973 | R²=0.7889
  Fold 3 | train=1,605 | val=620 | MAE=800,346 | RMSE=1,083,179 | R²=0.7703
  Fold 4 | train=2,225 | val=620 | MAE=458,220 | RMSE=634,774 | R²=0.7889
  Fold 5 | train=2,845 | val=623 | MAE=479,135 | RMSE=672,439 | R²=0.7698

OOF: MAE=661,510 | RMSE=956,909 | R²=0.8183



## Train Final Models (Full Train Data)

In [63]:
def get_xgb_params():
    return dict(
        n_estimators     = 2000,
        learning_rate    = 0.03,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        random_state     = 42,
        verbosity        = 0,
    )

def train_final(df_full: pd.DataFrame, feature_cols: list, target: str):
    df_full = df_full.dropna(subset=feature_cols + [target]).copy()
    X = df_full[feature_cols]
    y = df_full[target]
    print(f"  Training on {len(X):,} samples")

    # LightGBM
    lgb_model = lgb.LGBMRegressor(**get_lgb_params(target))
    lgb_model.fit(X, y, callbacks=[lgb.log_evaluation(period=-1)])
    print(f"  LGB best iteration: {lgb_model.best_iteration_}")

    # XGBoost
    xgb_model = xgb.XGBRegressor(**get_xgb_params())
    xgb_model.fit(X, y)
    print(f"  XGB n_estimators: {xgb_model.n_estimators}")

    return lgb_model, xgb_model

print("Training Revenue Models")
lgb_rev, xgb_rev = train_final(df_feat, feature_cols, "Revenue")

print()
print("Training COGS Models")
lgb_cogs, xgb_cogs = train_final(df_feat, feature_cols, "COGS")

Training Revenue Models
  Training on 3,468 samples
  LGB best iteration: 0
  XGB n_estimators: 2000

Training COGS Models
  Training on 3,468 samples
  LGB best iteration: 0
  XGB n_estimators: 2000


## Ensemble Weight Tuning (trên Holdout)

In [64]:
from scipy.optimize import minimize

def find_best_weights(lgb_preds, xgb_preds, y_true, metric="rmse"):
    def obj(w):
        w_norm = np.exp(w) / np.exp(w).sum()
        preds  = w_norm[0] * lgb_preds + w_norm[1] * xgb_preds
        if metric == "rmse":
            return np.sqrt(mean_squared_error(y_true, preds))
        return mean_absolute_error(y_true, preds)

    res = minimize(obj, [0.0, 0.0], method="Nelder-Mead",
                   options={"maxiter": 5000, "xatol": 1e-8})
    w = np.exp(res.x) / np.exp(res.x).sum()
    return float(w[0]), float(w[1])

def calc_metrics(y_true, y_pred, label):
    mae_  = mean_absolute_error(y_true, y_pred)
    rmse_ = np.sqrt(mean_squared_error(y_true, y_pred))
    r2_   = r2_score(y_true, y_pred)
    print(f"  [{label}]  MAE={mae_:,.0f} | RMSE={rmse_:,.0f} | R²={r2_:.4f}")

# Holdout = last 15% của train
df_clean = df_feat.dropna(subset=feature_cols + ["Revenue", "COGS"]).reset_index(drop=True)
val_cut  = int(len(df_clean) * 0.85)

X_val   = df_clean.iloc[val_cut:][feature_cols]
y_val_r = df_clean.iloc[val_cut:]["Revenue"]
y_val_c = df_clean.iloc[val_cut:]["COGS"]

print(f"Holdout size: {len(X_val):,} samples  "
      f"({df_clean.iloc[val_cut]['Date'].date()} - {df_clean.iloc[-1]['Date'].date()})")

# Predictions on holdout
lgb_val_r = lgb_rev.predict(X_val)
xgb_val_r = xgb_rev.predict(X_val)
lgb_val_c = lgb_cogs.predict(X_val)
xgb_val_c = xgb_cogs.predict(X_val)

# Individual model scores
print("\nIndividual model scores:")
calc_metrics(y_val_r, lgb_val_r, "LGB Revenue")
calc_metrics(y_val_r, xgb_val_r, "XGB Revenue")
calc_metrics(y_val_c, lgb_val_c, "LGB COGS")
calc_metrics(y_val_c, xgb_val_c, "XGB COGS")

# Optimize weights
w_lgb_r, w_xgb_r = find_best_weights(lgb_val_r, xgb_val_r, y_val_r.values, "rmse")
w_lgb_c, w_xgb_c = find_best_weights(lgb_val_c, xgb_val_c, y_val_c.values, "mae")

print(f"\nOptimal weights:")
print(f"  Revenue: LGB={w_lgb_r:.4f} | XGB={w_xgb_r:.4f}")
print(f"  COGS: LGB={w_lgb_c:.4f} | XGB={w_xgb_c:.4f}")

# Ensemble scores
ens_val_r = w_lgb_r * lgb_val_r + w_xgb_r * xgb_val_r
ens_val_c = w_lgb_c * lgb_val_c + w_xgb_c * xgb_val_c

print("\nEnsemble scores (holdout proxy for LB):")
calc_metrics(y_val_r, ens_val_r, "Ensemble Revenue")
calc_metrics(y_val_c, ens_val_c, "Ensemble COGS")

Holdout size: 521 samples  (2021-07-29 - 2022-12-31)

Individual model scores:
  [LGB Revenue]  MAE=5,744 | RMSE=8,103 | R²=1.0000
  [XGB Revenue]  MAE=23,165 | RMSE=30,623 | R²=0.9996
  [LGB COGS]  MAE=5,008 | RMSE=7,156 | R²=1.0000
  [XGB COGS]  MAE=20,388 | RMSE=26,872 | R²=0.9996

Optimal weights:
  Revenue: LGB=1.0000 | XGB=0.0000
  COGS: LGB=1.0000 | XGB=0.0000

Ensemble scores (holdout proxy for LB):
  [Ensemble Revenue]  MAE=5,744 | RMSE=8,103 | R²=1.0000
  [Ensemble COGS]  MAE=5,008 | RMSE=7,156 | R²=1.0000


Vấn đề nghiêm trọng: Holdout quá dễ!

MAE chỉ 5,744 trên holdout nhưng Kaggle score = 1,458,894 → holdout KHÔNG đại diện cho test. Lý do:

Final model train trên toàn bộ data bao gồm cả holdout → model đã "thấy" holdout rồi → MAE gần 0
Trọng số LGB=1.0 / XGB=0.0 có thể không tối ưu cho test thật
Fix: Dùng OOF predictions từ Walk-forward CV (Section 3) để tìm weights thay vì holdout từ final model.

## Build Test Features

In [65]:
def build_test_features(df_train_full, sample_sub, promotions, web, feature_cols):
    """
    Ghép train + test rows, build features, lấy phần test.
    Đảm bảo lag_365, lag_90, ... không bị NaN ở test period.
    """
    test_stub = sample_sub[["Date"]].copy()
    test_stub["Revenue"] = np.nan
    test_stub["COGS"]    = np.nan

    df_combined = pd.concat(
        [df_train_full[["Date", "Revenue", "COGS"]], test_stub],
        ignore_index=True
    )
    df_combined = build_features(df_combined, promotions, web)

    df_test = df_combined[df_combined["Date"] >= sample_sub["Date"].min()].copy()
    X_test  = df_test[feature_cols]
    return X_test, df_test

X_test, df_test = build_test_features(sales, sample_sub, promotions, web, feature_cols)

nan_count = X_test.isna().sum().sum()
print(f"Test shape: {X_test.shape}")
print(f"NaN count: {nan_count}")

if nan_count > 0:
    print("\nNaN per column:")
    nan_cols = X_test.isna().sum()
    print(nan_cols[nan_cols > 0])

    # Fill remaining NaN với median của train
    train_medians = df_clean[feature_cols].median()
    X_test = X_test.fillna(train_medians)
    print(f"\nAfter fillna: NaN = {X_test.isna().sum().sum()}")

print("\nTest features ready")

Test shape: (548, 61)
NaN count: 17225

NaN per column:
rev_lag_1           547
rev_lag_2           546
rev_lag_3           545
rev_lag_7           541
rev_lag_14          534
rev_lag_21          527
rev_lag_28          520
rev_lag_30          518
rev_lag_60          488
rev_lag_90          458
rev_lag_365         183
rev_roll7_mean      547
rev_roll7_std       547
rev_roll7_max       547
rev_roll7_min       547
rev_roll14_mean     547
rev_roll14_std      547
rev_roll14_max      547
rev_roll14_min      547
rev_roll30_mean     547
rev_roll30_std      547
rev_roll30_max      547
rev_roll30_min      547
rev_roll90_mean     547
rev_roll90_std      547
rev_roll90_max      547
rev_roll90_min      547
rev_yoy_lag365      183
cogs_lag_1          547
cogs_lag_7          541
cogs_lag_30         518
cogs_lag_365        183
cogs_roll7_mean     547
cogs_roll30_mean    547
dtype: int64

After fillna: NaN = 0

Test features ready


Vấn đề lớn nhất của v4: Test features bị NaN tràn lan!

rev_lag_1 = Revenue ngày hôm qua → Test bắt đầu từ 2023-01-01, chỉ có Revenue ngày 2022-12-31 (ngày cuối train) → lag_1 chỉ có giá trị cho ngày đầu test, 547 ngày còn lại = NaN
Tương tự cho tất cả lag features
Fill NaN bằng median là cách xử lý rất thô → Model mất đi thông tin quan trọng nhất (lag features)
Đây là nguyên nhân chính khiến Kaggle score cao (kém)!

## Generate Test Predictions

In [66]:
def ensemble_predict(lgb_model, xgb_model, X, w_lgb, w_xgb):
    p_lgb = lgb_model.predict(X)
    p_xgb = xgb_model.predict(X)
    return w_lgb * p_lgb + w_xgb * p_xgb

pred_rev = ensemble_predict(lgb_rev, xgb_rev,   X_test, w_lgb_r, w_xgb_r)
pred_cog = ensemble_predict(lgb_cogs, xgb_cogs, X_test, w_lgb_c, w_xgb_c)

pred_rev = np.round(np.clip(pred_rev, 0, None), 2)
pred_cog = np.round(np.clip(pred_cog, 0, None), 2)

print("Revenue predictions:")
print(f"min={pred_rev.min():,.0f} | mean={pred_rev.mean():,.0f} | max={pred_rev.max():,.0f}")
print("\nCOGS predictions:")
print(f"min={pred_cog.min():,.0f} | mean={pred_cog.mean():,.0f} | max={pred_cog.max():,.0f}")

Revenue predictions:
min=1,952,090 | mean=3,710,971 | max=7,010,728

COGS predictions:
min=1,562,412 | mean=3,101,250 | max=5,894,791


## 8. Save Submission

In [67]:
import os

sub = sample_sub.copy()
sub["Revenue"] = pred_rev
sub["COGS"] = pred_cog

assert len(sub) == 548, f"ERROR: expected 548 rows, got {len(sub)}"

out_path = f"../submissions/submission_v4.csv"
sub.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(sub)}")
sub.head(10)

Saved: ../submissions/submission_v4.csv
Rows: 548


,Date,Revenue,COGS
0,2023-01-01,1952090.31,1562412.08
1,2023-01-02,2538067.70,2079199.38
2,2023-01-03,2567672.41,1969556.71
3,2023-01-04,2368036.36,1839301.59
4,2023-01-05,2506598.05,2029061.13
5,2023-01-06,2953468.45,2518595.94
6,2023-01-07,2899770.07,2355509.25
7,2023-01-08,3316693.97,2755153.17
8,2023-01-09,3251248.07,2780626.11
9,2023-01-10,3387005.27,2933774.48


## Sanity Check & Visualization

In [68]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Revenue
ax = axes[0]
ax.plot(sales["Date"], sales["Revenue"], alpha=0.6, label="Train Revenue", color="steelblue")
ax.plot(sub["Date"], sub["Revenue"], color="tomato", linewidth=2, label="Predicted Revenue (v4)")
ax.set_title("Revenue: Train vs Prediction", fontsize=13)
ax.legend()
ax.set_ylabel("Revenue (VND)")
ax.grid(alpha=0.3)

# COGS
ax = axes[1]
ax.plot(sales["Date"], sales["COGS"], alpha=0.6, label="Train COGS", color="darkorange")
ax.plot(sub["Date"], sub["COGS"], color="purple", linewidth=2, label="Predicted COGS (v4)")
ax.set_title("COGS: Train vs Prediction", fontsize=13)
ax.legend()
ax.set_ylabel("COGS (VND)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"../report/figures/prediction_plot_v4.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved")

Plot saved


## Feature Importance (LightGBM)

In [69]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, title in zip(axes,
                             [lgb_rev, lgb_cogs],
                             ["Revenue", "COGS"]):
    importance = pd.Series(model.feature_importances_, index=feature_cols)
    top20 = importance.nlargest(20)
    top20[::-1].plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"LGB Feature Importance — {title}", fontsize=12)
    ax.set_xlabel("Importance")
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(f"../report/figures/feature_importance_v4.png", dpi=120, bbox_inches="tight")
plt.show()
print("Feature importance saved")

Feature importance saved
